# Módulo 3: Agentes de IA
- **Sesión:** 02 — Frameworks de agentes: LangGraph y protocolo MCP
- **Fecha:** 28/10/2026



---
## Fundamentos — El flujo como grafo



LangGraph representa una aplicación como un **grafo de estados**:

| Elemento | Qué es | En código |
|---|---|---|
| **Estado** | Diccionario compartido que viaja por el grafo | Una clase `TypedDict` |
| **Nodo** | Función que recibe el estado y devuelve las claves que actualiza | `grafo.add_node("nombre", funcion)` |
| **Arista** | Indica qué nodo se ejecuta después | `grafo.add_edge("a", "b")` |
| **Arista condicional** | Elige el siguiente nodo según el estado | `grafo.add_conditional_edges(...)` |

`START` y `END` son nodos especiales que marcan la entrada y la salida del grafo.


---
## Paso 0 — Configuración: modelo, datos y herramientas

### Concepto
`init_chat_model` crea el modelo a partir de `PREFIX_MODEL`. Las herramientas consultan los datos simulados de TiendaYa: son las mismas de la Sesión 1.

### Ejecutar

In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [2]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



In [ ]:
from langchain.tools import tool

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

POLITICAS = {
    "devoluciones": "15 días calendario desde la entrega; 7 días para productos electrónicos.",
    "envios": "Envío gratuito desde S/ 150; envío express en Lima por S/ 24.90.",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa. Usar cuando el cliente menciona un número de pedido."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")


@tool
def consultar_politica(tema: str) -> str:
    """Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'envios'."""
    return POLITICAS.get(tema.lower(), "No existe una política sobre ese tema.")


SYSTEM_PROMPT = ("Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos. "
                 "Si la información no proviene de una herramienta, indica que no la tienes. Responde de forma breve.")

print(consultar_pedido.invoke({"numero_pedido": "48213"}))


def ver_grafo(app):
    """Muestra el diagrama del grafo como imagen; si no hay conexión, lo imprime en formato Mermaid."""
    try:
        from IPython.display import Image, display
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

---
## Paso 1 — Primer grafo: el encadenamiento como grafo

### Concepto
Se construye el workflow de encadenamiento de la Sesión 1 como un grafo de tres nodos en secuencia:

```
START ──▶ extraer ──▶ consultar ──▶ redactar ──▶ END
            │            │            │
         numero    estado_pedido   respuesta      ← claves que agrega cada nodo
```

La construcción sigue siempre tres pasos:
1. **Estado:** se declaran las claves que comparten los nodos.
2. **Nodos:** cada función recibe el estado completo y devuelve **solo** las claves que modifica; LangGraph las combina con el estado anterior.
3. **Grafo:** se agregan los nodos y las aristas, y se **compila**. `compile()` valida la estructura y devuelve un objeto ejecutable.

### Ejecutar: estado y nodos

In [3]:
PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}


## Ejercicio 1 — Un grafo con un solo nodo
### Concepto
El grafo más simple tiene un nodo entre `START` y `END`. Se construye en tres pasos:

1. **Estado:** una clase `TypedDict` declara los datos del grafo.
2. **Nodo:** una función que recibe el estado y devuelve un diccionario **solo con lo que cambia**.
3. **Grafo:** se agregan el nodo y las aristas, y se **compila** con `compile()`.

```
START ──▶ saludar ──▶ END
```

### Ejecutar

In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# 1. Estado
class EstadoSaludo(TypedDict) :
    nombre : str
    saludo : str


# 2. Nodo 
def saludar(estado : EstadoSaludo ) -> dict:

    mensaje = f"""
          Hola {estado['nombre']} , Bienvenid@ a la TIENDA
        """
    return {"saludo": mensaje}

# 3. Grafo

grafo = StateGraph(EstadoSaludo)

# Agregar nodos
grafo.add_node("saludar",saludar)

# Agregar aristas
grafo.add_edge(START , "saludar")
grafo.add_edge("saludar", END)


app = grafo.compile()
res = app.invoke({"nombre": "Jaime"})

print(res)


{'nombre': 'Jaime', 'saludo': '\n          Hola Jaime , Bienvenid@ a la TIENDA\n        '}
